In [1]:
import os
import sys
maindir = os.getcwd()
sys.path.append(maindir+"/src")

In [2]:
import pickle
import torch
import numpy as np
import matplotlib.pyplot as plt

from preprocessing import data_processing, compute_anomalies_and_scalers, \
                            compute_forced_response, \
                            numpy_to_torch, rescale_and_merge_training_and_test_sets, \
                            rescale_training_and_test_sets


from plot_tools import plot_gt_vs_pred, animation_gt_vs_pred
from leave_one_out import leave_one_out_single, leave_one_out_procedure
from cross_validation import cross_validation_procedure


from algorithms import ridge_regression, ridge_regression_low_rank, \
                        train_trace_norm,train_robust_weights, compute_gradient,train_robust_weights_trace_norm,\
                        prediction, compute_weights, low_rank_projection

In [3]:
############### Load climate model raw data for SST
with open('data/ssp585_time_series.pkl', 'rb') as f:
    data = pickle.load(f)

###################### Load longitude and latitude 
with open('data/lon.npy', 'rb') as f:
    lon = np.load(f)

with open('data/lat.npy', 'rb') as f:
    lat = np.load(f)

# define grid (+ croping for latitude > 60)
lat_grid, lon_grid = np.meshgrid(lat[lat<=60], lon, indexing='ij')


lat_size = lat_grid.shape[0]
lon_size = lon_grid.shape[1]

In [24]:
# define pytorch precision
dtype = torch.float32

data_processed, notnan_idx, nan_idx = data_processing(data, lon, lat,max_models=100)
x, means, vars = compute_anomalies_and_scalers(data_processed, lon_size, lat_size, nan_idx, time_period=34)
y = compute_forced_response(data_processed, lon_size, lat_size, nan_idx, time_period=34)

x,y, means, vars = numpy_to_torch(x,y,means,vars, dtype=dtype)

############### REMOVE model 'GISS-E2-2-G' from the dataset ################
# x.pop('GISS-E2-2-G')
# y.pop('GISS-E2-2-G')
# means.pop('GISS-E2-2-G')
# vars.pop('GISS-E2-2-G')

/home/vcohen/cope/src/preprocessing.py:93: RuntimeWarning: Mean of empty slice
  means[m] = np.nanmean(data_reshaped[m],axis=1) - np.nanmean(data_reshaped[m],axis=(0,1))
/home/vcohen/cope/src/preprocessing.py:98: RuntimeWarning: Degrees of freedom <= 0 for slice.
  vars[m] = np.nanvar(data_reshaped[m],axis=(0,1))
/home/vcohen/cope/src/preprocessing.py:105: RuntimeWarning: Mean of empty slice
  data_reshaped[m] = data_reshaped[m] - np.expand_dims(np.nanmean(data_reshaped[m],axis=1),axis=1).repeat(time_period,axis=1)
/home/vcohen/cope/src/preprocessing.py:139: RuntimeWarning: Mean of empty slice
  mean_spatial_ensemble = np.nanmean(y_tmp,axis=0)
/home/vcohen/cope/src/preprocessing.py:143: RuntimeWarning: Mean of empty slice
  data_forced_response[m][idx_r,:,:] = mean_spatial_ensemble - np.nanmean(y_tmp,axis=(0,1))


In [25]:
m0 ='GISS-E2-2-G'
# m0 = 'EC-Earth3'
# m0 = 'GISS-E2-2-H'
training_models, x_rescaled, y_rescaled = rescale_training_and_test_sets(m0,x,y,means,vars,dtype=dtype)
training_models, x_train, y_train, x_test, y_test = rescale_and_merge_training_and_test_sets(m0,x,y,means,vars,dtype=dtype)


In [51]:
import geotorch
import torch.nn as nn

# class Model(torch.nn.Module):
#     def __init__(self):
#         super().__init__()
#         # One line suffices: Instantiate a linear layer with orthonormal columns
#         self.linear = nn.Linear(64, 128)
#         geotorch.orthogonal(self.linear, "weight")

#         # Works with tensors: Instantiate a CNN with kernels of rank 1
#         self.cnn = nn.Conv2d(16, 32, 3)
#         geotorch.low_rank(self.cnn, "weight", rank=1)

#         # Weights are initialized to a random value when you put the constraints, but
#         # you may re-initialize them to a different value by assigning to them
#         self.linear.weight = torch.eye(128, 64)
#         # And that's all you need to do. The rest is regular PyTorch code

#     def forward(self, x):
#         # self.linear is orthogonal and every 3x3 kernel in self.cnn is of rank 1
#         return self.linear(x)

# lr=1e-3
# # Use the model as you would normally do. Everything just works
# model = Model()

# # Use your optimizer of choice. Any optimizer works out of the box with any parametrization
# optim = torch.optim.Adam(model.parameters(), lr=lr)

In [63]:
class LowRankModel(nn.Module):
    def __init__(self, input_dim, output_dim, rank):
        super().__init__()
        self.linear = nn.Linear(input_dim, output_dim, bias=False)
        # Apply a low-rank constraint to the weight matrix
        geotorch.low_rank(self.linear, "weight", rank=rank)

    def forward(self, x):
        return self.linear(x)

def compute_mse_loss(m, x, y):
    output = model(x[m])
    loss = torch.nn.functional.mse_loss(output, y[m], model.linear)
    return loss

def list_mse_loss(model,x, y):
    loss = torch.zeros(len(training_models))
    for idx_m, m in enumerate(training_models):
        output = model(x[m][:,:,notnan_idx])
        loss[idx_m] = torch.nn.functional.mse_loss(output, y[m][:,:,notnan_idx])
    return loss

# Define the function to optimize
def loss_function(output, target, layer):
    return torch.logsumexp((1/100.0)*compute_mse_loss(output, target)) + 10*torch.sum(layer.weight**2)

# Initialize the model, optimizer, and data
input_dim = len(notnan_idx)
output_dim = len(notnan_idx)
rank = 50
model = LowRankModel(input_dim, output_dim, rank)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


# Training loop
num_epochs = 100
for epoch in range(num_epochs):
    optimizer.zero_grad()
    # output = model(x[:, notnan_idx])
    loss = list_mse_loss(model,x,y)
    print(loss.shape)
    loss = 100*torch.logsumexp((1/100.0)*loss,0) + 10*torch.sum(model.linear.weight**2)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss.item()}")
    # Backpropagation
    # Compute gradients
    loss.backward()
    # Update weights
    optimizer.step()

torch.Size([28])
Epoch 1/100, Loss: 2109.05029296875
torch.Size([28])
Epoch 2/100, Loss: 2107.167236328125
torch.Size([28])
Epoch 3/100, Loss: 2105.272216796875
torch.Size([28])
Epoch 4/100, Loss: 2103.389404296875
torch.Size([28])
Epoch 5/100, Loss: 2101.4970703125
torch.Size([28])
Epoch 6/100, Loss: 2099.595703125


In [50]:
input = x_train[:,notnan_idx]
output = layer(input)